# Notebook 05 — Training Data Prep, S3 Upload, and Flywheel Management

This notebook demonstrates the training data pipeline:
1. Preparing annotation CSVs in Comprehend format
2. Uploading documents and annotations to S3
3. Creating and managing a Comprehend Flywheel
4. Triggering a training iteration and polling its status

**What this covers:**
- `prepare_annotations()`: sentences + entities → Comprehend annotation CSV format
- `upload_documents()` and `upload_annotations()`: S3 upload flow
- `create_flywheel()`, `update_flywheel_iteration()`, `poll_flywheel_status()`

**Why Flywheel:** Amazon Comprehend Flywheel automates the retraining cycle.
When new labelled data arrives (via the feedback loop), we trigger an iteration
and Flywheel handles model training, evaluation, and deployment.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from unittest.mock import patch, MagicMock

## 1. Annotation CSV Format

Amazon Comprehend expects training annotations in a CSV with exactly these columns:

| Column | Description |
|--------|-------------|
| File | Filename of the corresponding document |
| Line | 0-based line number within the document |
| Begin Offset | Character offset of entity start (0-based) |
| End Offset | Character offset one past the entity end |
| Type | Entity type label |

One row per entity annotation.  Multiple entities in the same sentence each
get their own row referencing the same `File` and `Line`.

In [ ]:
# Mock sentences and entities.
mock_sentences = [
    {"row_index": 0, "text": "JOHN DOE was added to the sanctions list.", "filename": "doc_0.txt"},
    {"row_index": 1, "text": "The vessel DARK STAR was seized.", "filename": "doc_1.txt"},
    {"row_index": 2, "text": "ACME CORP funded IVAN PETROV activities.", "filename": "doc_2.txt"},
]

mock_entities = [
    {"row_index": 0, "entity_text": "JOHN DOE",   "entity_type": "SANCTIONED_PERSON", "BeginOffset": 0,  "EndOffset": 8,  "line": 0},
    {"row_index": 1, "entity_text": "DARK STAR",  "entity_type": "SANCTIONED_ORG",    "BeginOffset": 11, "EndOffset": 20, "line": 0},
    {"row_index": 2, "entity_text": "ACME CORP",  "entity_type": "SANCTIONED_ORG",    "BeginOffset": 0,  "EndOffset": 9,  "line": 0},
    {"row_index": 2, "entity_text": "IVAN PETROV","entity_type": "SANCTIONED_PERSON", "BeginOffset": 17, "EndOffset": 28, "line": 0},
]

print(f"Sentences: {len(mock_sentences)}")
print(f"Entity annotations: {len(mock_entities)}")

In [ ]:
from ner_ofac.training.annotation_prep import prepare_annotations

annotation_df = prepare_annotations(mock_sentences, mock_entities)

print("Annotation DataFrame:")
annotation_df

In [ ]:
# Verify entity text against offsets.
print("Verifying entity spans:")
sentence_map = {s['filename']: s['text'] for s in mock_sentences}
for _, row in annotation_df.iterrows():
    text = sentence_map.get(row['File'], '')
    span = text[row['Begin Offset']:row['End Offset']]
    print(f"  File={row['File']}, Type={row['Type']}: '{span}'")

## 2. Uploading to S3

`upload_documents()` uploads a local directory of plain-text documents.
`upload_annotations()` serializes the annotation DataFrame to CSV and uploads it.

We mock `boto3.client` here to avoid real S3 calls.

In [ ]:
import tempfile

# Create mock document files.
with tempfile.TemporaryDirectory() as doc_dir:
    for sent in mock_sentences:
        fpath = os.path.join(doc_dir, sent['filename'])
        with open(fpath, 'w') as f:
            f.write(sent['text'])
    
    print(f"Mock documents created in: {doc_dir}")
    print(os.listdir(doc_dir))
    
    # Mock S3 upload.
    mock_s3 = MagicMock()
    with patch('boto3.client', return_value=mock_s3):
        from ner_ofac.training.upload_to_s3 import upload_documents, upload_annotations
        s3_docs_uri = upload_documents(doc_dir, s3_bucket='my-bucket', s3_prefix='training/documents')
        print(f"\nDocuments URI: {s3_docs_uri}")
        
        s3_annot_uri = upload_annotations(annotation_df, s3_bucket='my-bucket', s3_prefix='training/annotations')
        print(f"Annotations URI: {s3_annot_uri}")

## 3. Flywheel Management

The Flywheel automates retraining.  We create it once and then call
`update_flywheel_iteration()` whenever new annotated data is available.

In [ ]:
from ner_ofac.training.flywheel_manager import (
    create_flywheel,
    update_flywheel_iteration,
    poll_flywheel_status,
    activate_model_version,
)

# Mock Comprehend client responses.
mock_comprehend = MagicMock()
mock_comprehend.create_flywheel.return_value = {
    'FlywheelArn': 'arn:aws:comprehend:us-east-1:123456:flywheel/ner-ofac-flywheel'
}
mock_comprehend.create_flywheel_iteration.return_value = {
    'FlywheelIterationProperties': {
        'FlywheelIterationArn': 'arn:aws:comprehend:us-east-1:123456:flywheel/ner-ofac-flywheel/iter-001'
    }
}
# Simulate TRAINING → COMPLETED transition.
status_responses = iter([
    {'FlywheelIterationProperties': {'Status': 'TRAINING'}},
    {'FlywheelIterationProperties': {'Status': 'COMPLETED'}},
])
mock_comprehend.describe_flywheel_iteration.side_effect = lambda **kw: next(status_responses)

with patch('boto3.client', return_value=mock_comprehend), \
     patch('time.sleep'):  # Skip actual sleep in poll loop.
    
    # Create a new Flywheel.
    flywheel_arn = create_flywheel(
        name='ner-ofac-flywheel',
        data_lake_s3_uri='s3://my-bucket/flywheel-data-lake/'
    )
    print(f"Flywheel ARN: {flywheel_arn}")
    
    # Trigger a new training iteration.
    training_config = {'S3Uri': 's3://my-bucket/training/annotations/'}
    iteration_arn = update_flywheel_iteration(flywheel_arn, training_config)
    print(f"Iteration ARN: {iteration_arn}")
    
    # Poll until complete.
    final_status = poll_flywheel_status(flywheel_arn, iteration_arn, poll_interval=1)
    print(f"Final status: {final_status}")

## Summary

- `prepare_annotations()` produces Comprehend-compatible annotation CSV format
- Annotations are uploaded to S3 alongside plain-text documents
- `create_flywheel()` sets up the continuous training loop (run once)
- `update_flywheel_iteration()` triggers a new training cycle with fresh data
- `poll_flywheel_status()` waits for training to complete before activating